<div dir="rtl">

# ۰۱ — خواندن، ادغام و ساخت نمونه‌ی تحلیلی

**پروژه:** وضعیت مصرف سیگار و مدت خواب در بزرگسالان آمریکایی (NHANES ۲۰۲۱–۲۰۲۳)

**هدف این نوت‌بوک:**
1. خواندن سه فایل خام `DEMO_L`، `SLQ_L` و `SMQ_L`
2. بررسی واحد مشاهده: آیا هر نفر فقط یک ردیف دارد؟
3. ادغام سه فایل با کلید `SEQN` و گزارش تعداد افراد قبل و بعد از ادغام (تسک الف.۱)
4. ساخت متغیرهای مشتق: وضعیت سیگار، میانگین هفتگی خواب، خواب کوتاه
5. ساخت جمعیت تحلیلی و جدول حذف افراد در هر مرحله (تسک الف.۴)
6. ذخیره‌ی داده‌ی تحلیلی برای نوت‌بوک‌های بعدی

پیش‌نیاز: فایل‌های خام با `python src/download_data.py` دانلود شده باشند.

</div>

In [2]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("..").resolve() / "src"))
import nhanes as nh

pd.set_option("display.width", 160)

<div dir="rtl">

## ۱. خواندن فایل‌های خام

تابع `nh.load_raw` هر فایل `.xpt` را می‌خواند و فقط ستون‌هایی را نگه می‌دارد که در پروژه لازم داریم.

</div>

In [3]:
demo = nh.load_raw("DEMO_L")
slq = nh.load_raw("SLQ_L")
smq = nh.load_raw("SMQ_L")

for name, df in [("DEMO_L", demo), ("SLQ_L", slq), ("SMQ_L", smq)]:
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

DEMO_L: 11,933 rows, 6 columns
SLQ_L: 8,501 rows, 3 columns
SMQ_L: 9,015 rows, 4 columns


In [4]:
demo.head()

,SEQN,RIAGENDR,RIDAGEYR,WTINT2YR,SDMVPSU,SDMVSTRA
0,130378,1.0,43.0,50055.450807,2.0,173.0
1,130379,1.0,66.0,29087.450605,2.0,173.0
2,130380,2.0,44.0,80062.674301,1.0,174.0
3,130381,2.0,5.0,38807.268902,2.0,182.0
4,130382,1.0,2.0,30607.519774,2.0,182.0


In [5]:
slq.head()

,SEQN,SLD012,SLD013
0,130378,9.5,9.0
1,130379,9.0,9.0
2,130380,8.0,9.0
3,130384,7.5,8.0
4,130385,8.0,8.0


In [6]:
smq.head()

,SEQN,SMQ020,SMQ040,SMD650
0,130378,1.0,3.0,NaN
1,130379,1.0,3.0,NaN
2,130380,2.0,NaN,NaN
3,130384,2.0,NaN,NaN
4,130385,2.0,NaN,NaN


<div dir="rtl">

## ۲. واحد مشاهده: هر ردیف یعنی چه؟

اگر هر نفر فقط یک ردیف داشته باشد، تعداد ردیف‌ها با تعداد `SEQN` یکتا برابر است.

</div>

In [7]:
for name, df in [("DEMO_L", demo), ("SLQ_L", slq), ("SMQ_L", smq)]:
    print(f"{name}: rows = {len(df):,}, unique SEQN = {df['SEQN'].nunique():,}, "
          f"one row per person: {len(df) == df['SEQN'].nunique()}")

DEMO_L: rows = 11,933, unique SEQN = 11,933, one row per person: True
SLQ_L: rows = 8,501, unique SEQN = 8,501, one row per person: True
SMQ_L: rows = 9,015, unique SEQN = 9,015, one row per person: True


<div dir="rtl">

**نتیجه:** در هر سه فایل، هر ردیف دقیقاً یک شرکت‌کننده است و هر نفر یک بار مشاهده شده (مطالعه‌ی مقطعی).

تعداد ردیف‌ها در سه فایل متفاوت است، چون هر پرسش‌نامه از یک گروه سنی پرسیده شده:
- `DEMO_L`: همه‌ی شرکت‌کننده‌ها
- `SLQ_L`: ۱۶ سال و بالاتر
- `SMQ_L`: ۱۲ سال و بالاتر

</div>

<div dir="rtl">

## ۳. ادغام سه فایل (تسک الف.۱)

سلام`DEMO_L` پایه است، چون همه‌ی افراد و وزن‌های نمونه‌گیری در آن هستند. خواب و سیگار با **left join** روی `SEQN` به آن اضافه می‌شوند. یعنی هیچ‌کس از `DEMO_L` حذف نمی‌شود؛ اگر کسی در فایل خواب یا سیگار نباشد، آن ستون‌ها برایش خالی می‌مانند.

`validate="one_to_one"` در تابع ادغام تضمین می‌کند اگر `SEQN` تکراری وجود داشت، کد با خطا متوقف شود.

</div>

In [8]:
merged = nh.merge_files(demo, slq, smq)

print(f"Rows before merge (DEMO_L): {len(demo):,}")
print(f"Rows after merge:           {len(merged):,}")
print(f"People with sleep data:     {merged['SLD012'].notna().sum():,}")
print(f"People in smoking file:     {merged['SEQN'].isin(smq['SEQN']).sum():,}")

Rows before merge (DEMO_L): 11,933
Rows after merge:           11,933
People with sleep data:     8,388
People in smoking file:     9,015


<div dir="rtl">

## ۴. ساخت متغیرهای مشتق

| متغیر | تعریف |
|---|---|
| `smoking_status` | هرگز: `SMQ020`=۲ / سابق: `SMQ020`=۱ و `SMQ040`=۳ / فعلی: `SMQ020`=۱ و `SMQ040`=۱ یا ۲ |
| `sleep_weekly` | (۵ × `SLD012` + ۲ × `SLD013`) ÷ ۷ |
| `short_sleep` | ۱ اگر `sleep_weekly` کمتر از ۷ ساعت باشد، وگرنه ۰ |
| `sleep_coded_extreme` | آیا یکی از دو مقدار خواب کد ۲ (کمتر از ۳ ساعت) یا ۱۴ (۱۴ ساعت یا بیشتر) است؟ |

</div>

In [9]:
merged = nh.add_derived_variables(merged)

# Check the smoking classification against the raw codes
pd.crosstab(merged["SMQ020"].fillna(-1), merged["SMQ040"].fillna(-1),
            rownames=["SMQ020"], colnames=["SMQ040"], margins=True)

SMQ040,-1.0,1.0,2.0,3.0,All
SMQ020,,,,,
-1.0,3798,0,0,0,3798
1.0,0,952,238,2053,3243
2.0,4878,0,0,0,4878
7.0,7,0,0,0,7
9.0,7,0,0,0,7
All,8690,952,238,2053,11933


<div dir="rtl">

در جدول بالا `-1` یعنی خالی (پرسیده نشده یا گمشده). بررسی کن که:
- `SMQ020`=۲ همیشه `SMQ040` خالی دارد (سؤال دوم پرسیده نشده)
- کدهای ۷ و ۹ در `SMQ020` جزو هیچ گروهی نمی‌شوند

</div>

In [10]:
merged["smoking_status"].value_counts(dropna=False)

smoking_status
never      4878
NaN        3812
former     2053
current    1190
Name: count, dtype: int64

<div dir="rtl">

## ۵. جمعیت تحلیلی و جدول حذف افراد (تسک الف.۴)

معیارهای ورود به ترتیب اعمال می‌شوند و بعد از هر مرحله تعداد افراد باقی‌مانده ثبت می‌شود.

</div>

In [11]:
analytic, flow = nh.build_analytic_sample(merged)
flow

,step,n_remaining,n_excluded
0,All NHANES 2021-2023 participants,11933,0
1,Aged 18 or older,8153,3780
2,Valid smoking status (excludes refused / don't...,8121,32
3,Both weekday and weekend sleep hours available,7996,125


In [12]:
analytic["smoking_status"].value_counts()

smoking_status
never      4823
former     2020
current    1153
Name: count, dtype: int64

In [13]:
print(f"People in analytic sample with sleep code 2 or 14: {analytic['sleep_coded_extreme'].sum()}")

People in analytic sample with sleep code 2 or 14: 104


<div dir="rtl">

**تصمیم:** افرادی که کد ۲ یا ۱۴ دارند **حذف نمی‌شوند**. این کدها مقدار واقعی را دقیقاً نشان نمی‌دهند، ولی اطلاعات معتبر دارند (خواب بسیار کم یا بسیار زیاد). تعدادشان در بخش بررسی کیفیت داده گزارش می‌شود و در فاز پنجم، تحلیل یک بار بدون آن‌ها هم تکرار می‌شود.

</div>

<div dir="rtl">

## ۶. ذخیره‌ی داده‌ی تحلیلی

این فایل در `data/processed/` ذخیره می‌شود و به GitHub نمی‌رود. با اجرای دوباره‌ی همین نوت‌بوک ساخته می‌شود.

</div>

In [14]:
nh.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
analytic.to_csv(nh.PROCESSED_DIR / "analytic_sample.csv", index=False)
flow.to_csv(nh.PROCESSED_DIR / "flow_table.csv", index=False)
print(f"Saved {len(analytic):,} rows to {nh.PROCESSED_DIR / 'analytic_sample.csv'}")

Saved 7,996 rows to D:\University\Semaster-7\Statics And Its Aplication\Project\Statistics-Project-Fall2026\data\processed\analytic_sample.csv


<div dir="rtl">

## جمع‌بندی

- هر ردیف = یک شرکت‌کننده؛ `SEQN` در هر سه فایل یکتاست.
- ادغام با left join روی `DEMO_L` انجام شد و هیچ ردیفی اضافه یا کم نشد.
- جمعیت تحلیلی: بزرگسالان ۱۸+ با وضعیت سیگار معتبر و هر دو مقدار خواب.
- جدول حذف افراد در `flow` آمده و در گزارش فاز اول استفاده می‌شود.

</div>